# Three RCP pathways, many possible futures

**Atlas source reading copy · browser-network**

[Course guide](../index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Sea-Level-Rise/blob/66065673b47f9bc5c0f6e5009ac1c924a9f5200c/content/notebooks/02_three_rcp_pathways.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `content/notebooks/02_three_rcp_pathways.ipynb` at `66065673b47f9bc5c0f6e5009ac1c924a9f5200c`; original SHA-256 `df34161259006cf73124048a6f0819104fb33cfd128fe0ada66aa4f3681c1afa`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# Three RCP pathways, many possible futures

**Time:** 35 minutes · **Prerequisites:** basic Python; run cells from top to bottom.

Distinguish a forcing pathway from a model; interpret baselines and intervals; compare scenarios without treating them as probabilities.

[Run this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/02_three_rcp_pathways.ipynb) · [Full-screen dashboard](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/dashboard/) · [Official poster](https://plus.iasociety.org/sites/default/files/2024-09/e-poster_755.pdf)

Each notebook runs independently. Use **Run → Run All Cells**; the first package download can take a minute. Save a downloaded copy to keep your work. The read-only book includes executed examples.

In [ ]:
# First run downloads Python packages in JupyterLite; subsequent runs use browser caches.
import sys
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'plotly', 'ipywidgets'])
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML
from coastlab import (DATA, SCENARIOS, YEARS, COLORS, load_data, select, cards,
                      toy_slr, toy_probability, cumulative_probability,
                      haversine_km, show, dashboard, land_traces)
facilities, exposure = load_data()
published = pd.read_csv(DATA / 'poster_aggregates.csv')
print('Ready: public poster aggregates + clearly labeled fictional facility fixtures.')

## What the numbers mean

RCP 2.6, RCP 4.5 and RCP 8.5 label representative radiative-forcing pathways, conventionally expressed in W/m² near 2100. They are not degrees of warming, metres of sea-level rise or probabilities. Multiple physical models can be run under the same pathway. RCP 8.5 is a high-forcing scenario, not an assertion that it is the most likely future.

Modern SSP-based assessments add a socioeconomic framework. SSP1-2.6 is not an interchangeable replacement for RCP 2.6: model generation, baselines and assumptions also matter. The book keeps the poster's historical RCP framework and links to [IPCC AR6](https://www.ipcc.ch/report/ar6/wg1/chapter/summary-for-policymakers/) for context.

## Read a historical assessment carefully

The reference CSV below transcribes six rows of **IPCC AR5 WGI Table SPM.2**: global mean sea-level change relative to **1986–2005**, averaged over **2046–2065** or **2081–2100**. These are period averages, not exact-year local predictions. They are educational context, not inputs used to recreate the poster's PAT calculations.

In [ ]:
ref = pd.read_csv(DATA / 'ipcc_ar5_reference.csv')
display(ref)
late = ref.query("period == '2081-2100'")
fig = go.Figure()
for _,r in late.iterrows():
    fig.add_trace(go.Scatter(x=[r.scenario],y=[r.mean_m],mode='markers',name=r.scenario,
        marker=dict(size=14,color=COLORS[r.scenario]),
        error_y=dict(type='data',symmetric=False,array=[r.likely_high_m-r.mean_m],arrayminus=[r.mean_m-r.likely_low_m])))
fig.update_layout(title='IPCC AR5: global change, 2081–2100 average',yaxis_title='Metres relative to 1986–2005',showlegend=False)
show(fig)

The ranges overlap. A pathway difference is not the only uncertainty: ice dynamics, ocean change, land motion and elevation errors can matter locally. A Bangkok facility cannot be classified by comparing its height to a global average unless the vertical datums, local processes and flood processes are made consistent.

## A deliberately simple teaching curve

For interactive labs we invent a 2020 baseline and endpoints of 0.45, 0.65 and 1.00 m in 2100. With $t=(y-2020)/80$, the curve is $0.2t+(H-0.2)t^2$. These chosen values are **not** the AR5 table, a fitted forecast, Kopp outputs or Climate Central data.

In [ ]:
toy = pd.DataFrame([{'scenario':s,'year':y,'toy_slr_m':toy_slr(s,y)} for s in SCENARIOS for y in YEARS])
fig=px.line(toy,x='year',y='toy_slr_m',color='scenario',markers=True,color_discrete_map=COLORS,
            title='Invented teaching curves — fictional 2020 baseline')
show(fig)

## Lab: compare like with like

Write a function that rejects combining projection tables when their baseline or period differs. Then explain why the near-term ordering in the published poster is 108, 107 and 108 in 2030. Do not force the historical data to be monotonic across scenarios just because the teaching curves are.

**Worked interpretation:** Scenario differences can be small near-term and do not override source-specific uncertainties and processing. The published 107 should remain 107. Explain the difference rather than replacing it with 108.

**Deliverable:** A labeled interval plot and a caption naming the source, vertical unit, baseline and time window.

Reference: [IPCC AR5 WGI Summary for Policymakers, Table SPM.2](https://www.ipcc.ch/site/assets/uploads/2018/02/WG1AR5_SPM_FINAL.pdf).